In [1]:
### Created by Myongin Oh
### Last updated on Aug 23, 2025

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_selection import mutual_info_classif
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

In [2]:
# Threshold for high correlation
threshold = 0.9

In [3]:
df1 = pd.read_csv('../../../feat_sel/residues/3_bpso/BPSO_07_iter.csv')
df2 = pd.read_csv('../../../feat_sel/residues/0_feature_calculation_selection_var_fisher/dfFinal_1766feats.csv')

zeroList = [0]*10000 # class 1y
oneList = [1]*10000 # class 2
twoList = [2]*10000 # class 3

A = df1
B = df2
y = np.array(zeroList + oneList + twoList)

In [6]:
# Standardize features for Pearson correlation coefficients
A_std = (A - A.mean()) / A.std()
B_std = (B - B.mean()) / B.std()

In [7]:
# Compute correlation matrix: shape (n_features_A, n_features_B)
cor_matrix = np.dot(A_std.T.values, B_std.values) / (len(A) - 1)

In [9]:
# Compute MI of A features with the class
mi_A = mutual_info_classif(A, y, discrete_features='auto') # discrete_features=False (treat all features continuous)

In [11]:
# Compare mutual info and select best feature (A or correlated B)
best_feature_names = []
for i, a_feat in enumerate(A.columns):
    # Step 1: correlated B features
    correlations = cor_matrix[i, :]
    high_corr_indices = np.where(np.abs(correlations) >= threshold)[0]

    if len(high_corr_indices) == 0:
        # No highly correlated B features → keep A feature
        best_feature_names.append(a_feat)

    else:
        # Step 2: compute MI for high-correlation B features
        B_high_corr = B.iloc[:, high_corr_indices]
        mi_B = mutual_info_classif(B_high_corr, y, discrete_features='auto')
        
        # Step 3: compare and select best
        max_mi_B = np.max(mi_B)
        if mi_A[i] >= max_mi_B:
            best_feature_names.append(a_feat)
        else:
            best_idx = high_corr_indices[np.argmax(mi_B)]
            best_feature_names.append(B.columns[best_idx])

In [12]:
# Extract selected features from B
C = B[best_feature_names]
C['class'] = y

C:\Users\myongino\AppData\Local\Temp\ipykernel_38052\1675786484.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  C['class'] = y


In [13]:
# Save to file
C.to_csv("final_bpso_mi.csv", index=False)